# 01 - Data cleaning & the SQL layer (retail churn)

**Goal:** turn the raw online-retail customer file (15 columns, one row per customer) into a typed,
validated analytical table - and make the cleaning rules auditable by writing them in SQL.

Design choice: the raw CSV is loaded into SQLite *as text*. All typing and cleaning happens in
`sql/01_create_schema.sql`, and `sql/02_data_quality_checks.sql` gates the pipeline. The same SQL later
cleans any file uploaded to the dashboard, so training and scoring can never drift apart.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src import configs  # save configs.py as src/config.py
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

In [2]:
from src.data_preprocessing import load_raw_csv, build_database, run_quality_checks, run_named_query
raw = load_raw_csv()
missing_cols = set(configs.RAW_REQUIRED_COLUMNS) - set(raw.columns)
assert not missing_cols, f"raw file is missing columns: {missing_cols}"
print(raw.shape)
raw.head()

(1000, 15)


,Customer_ID,Age,Gender,Annual_Income,Total_Spend,Years_as_Customer,Num_of_Purchases,Average_Transaction_Amount,Num_of_Returns,Num_of_Support_Contacts,Satisfaction_Score,Last_Purchase_Days_Ago,Email_Opt_In,Promotion_Response,Target_Churn
0,1,62,Other,45.15,5892.58,5,22,453.8,2,0,3,129,True,Responded,True
1,2,65,Male,79.51,9025.47,13,77,22.9,2,2,3,227,False,Responded,False
2,3,18,Male,29.19,618.83,13,71,50.53,5,2,2,283,False,Responded,True
3,4,21,Other,79.63,9110.3,3,33,411.83,5,3,5,226,True,Ignored,True
4,5,21,Other,77.66,5390.88,15,43,101.19,3,0,5,242,False,Unsubscribed,False


## Finding 1 - missing and blank values

In [3]:
def is_blank(s):
    return s.isna() | (s.astype(str).str.strip().isin(["", "nan", "NaN", "None", "NA", "N/A"]))

blanks = raw.apply(is_blank).sum().rename("blank_or_null").to_frame()
blanks["pct"] = (blanks["blank_or_null"] / len(raw) * 100).round(2)
blanks[blanks["blank_or_null"] > 0]

,blank_or_null,pct


*Interpretation (fill in after running):* are the blanks random, or tied to a pattern (e.g. a
group of customers with no purchases)? Decide the fix per column from the pattern, not a blanket median.

## Finding 2 - duplicates and categories

In [4]:
print("duplicate Customer_IDs:", raw["Customer_ID"].duplicated().sum())
print("fully duplicated rows:  ", raw.duplicated().sum())
for col in ["Gender", "Promotion_Response", "Email_Opt_In", "Target_Churn"]:
    print(f"{col:<20}", raw[col].value_counts(dropna=False).to_dict())

duplicate Customer_IDs: 0
fully duplicated rows:   0
Gender               {'Female': 342, 'Male': 334, 'Other': 324}
Promotion_Response   {'Unsubscribed': 361, 'Responded': 338, 'Ignored': 301}
Email_Opt_In         {'True': 529, 'False': 471}
Target_Churn         {'True': 526, 'False': 474}


Check for inconsistent spellings/casing (`male` vs `Male`), and make sure `Email_Opt_In` and
`Target_Churn` use one encoding. The SQL should map them to a single clean form.

## Finding 3 - impossible values and divide-by-zero risks

Several engineered features divide by `num_of_purchases`, `years_as_customer` or `annual_income`
(`return_rate`, `purchases_per_year`, `spend_per_year`, `support_contact_rate`, `rescency_ratio`,
`spend_to_income_ratio`). A zero in any of them produces `inf`/`NaN`, so look for them now.

In [5]:
num_cols = [c for c in configs.RAW_REQUIRED_COLUMNS
            if c not in ("Customer_ID", "Gender", "Email_Opt_In", "Promotion_Response", "Target_Churn")]
nums = raw[num_cols].apply(pd.to_numeric, errors="coerce")

rows = []
for c in num_cols:
    unparsable = nums[c].isna().sum() - is_blank(raw[c]).sum()
    rows.append({"column": c, "unparsable_text": unparsable, "negative": (nums[c] < 0).sum(),
                 "zero": (nums[c] == 0).sum(), "min": nums[c].min(), "max": nums[c].max()})
pd.DataFrame(rows)

,column,unparsable_text,negative,zero,min,max
0,Age,0,0,0,18.00,69.00
1,Annual_Income,0,0,0,20.01,199.73
2,Total_Spend,0,0,0,108.94,9999.64
3,Years_as_Customer,0,0,0,1.00,19.00
4,Num_of_Purchases,0,0,0,1.00,99.00
5,Average_Transaction_Amount,0,0,0,10.46,499.57
6,Num_of_Returns,0,0,102,0.00,9.00
7,Num_of_Support_Contacts,0,0,203,0.00,4.00
8,Satisfaction_Score,0,0,0,1.00,5.00
9,Last_Purchase_Days_Ago,0,0,0,1.00,364.00


In [6]:
# logical consistency checks
checks = {
    "last purchase older than customer relationship": (nums["Last_Purchase_Days_Ago"] > nums["Years_as_Customer"] * 365).sum(),
    "has purchases but zero total spend":             ((nums["Num_of_Purchases"] > 0) & (nums["Total_Spend"] == 0)).sum(),
    "returns exceed purchases":                       (nums["Num_of_Returns"] > nums["Num_of_Purchases"]).sum(),
    "avg transaction x purchases far from total (>2x off)":
        (((nums["Average_Transaction_Amount"] * nums["Num_of_Purchases"]) / nums["Total_Spend"]).between(0.5, 2) == False).sum(),
}
pd.Series(checks, name="rows")

last purchase older than customer relationship            0
has purchases but zero total spend                        0
returns exceed purchases                                 36
avg transaction x purchases far from total (>2x off)    689
Name: rows, dtype: int64

In [7]:
print("satisfaction_score values:", sorted(nums["Satisfaction_Score"].dropna().unique())[:15])
nums.describe().T.round(2)

satisfaction_score values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


,count,mean,std,min,25%,50%,75%,max
Age,1000.0,43.27,15.24,18.00,30.00,43.00,56.00,69.00
Annual_Income,1000.0,111.96,52.84,20.01,67.80,114.14,158.45,199.73
Total_Spend,1000.0,5080.79,2862.12,108.94,2678.68,4986.20,7606.47,9999.64
Years_as_Customer,1000.0,9.73,5.54,1.00,5.00,9.00,14.00,19.00
Num_of_Purchases,1000.0,49.46,28.54,1.00,25.00,49.00,74.00,99.00
Average_Transaction_Amount,1000.0,266.88,145.87,10.46,139.68,270.10,401.60,499.57
Num_of_Returns,1000.0,4.61,2.90,0.00,2.00,5.00,7.00,9.00
Num_of_Support_Contacts,1000.0,1.93,1.40,0.00,1.00,2.00,3.00,4.00
Satisfaction_Score,1000.0,2.97,1.39,1.00,2.00,3.00,4.00,5.00
Last_Purchase_Days_Ago,1000.0,182.89,104.39,1.00,93.00,180.50,274.00,364.00


*Interpretation (fill in after running):* for each issue, decide fix / drop / keep and write the rule
into `sql/01_create_schema.sql`. Investigate before dropping - a flagged row is often a real account.

## The cleaning SQL

In [8]:
print((configs.SQL_DIR / "01_create_schema.sql").read_text())

-- Retail customer churn schema matching src/configs.py.
-- Requires raw_customers to be loaded from configs.RAW_CSV first.

DROP VIEW IF EXISTS customer_features;
DROP TABLE IF EXISTS customers;

CREATE TABLE customers AS
SELECT
    NULLIF(TRIM(Customer_ID), '') AS customer_id,
    CAST(NULLIF(TRIM(Age), '') AS INTEGER) AS age,
    NULLIF(TRIM(Gender), '') AS gender,
    CAST(NULLIF(TRIM(Annual_Income), '') AS REAL) AS annual_income,
    CAST(NULLIF(TRIM(Total_Spend), '') AS REAL) AS total_spend,
    CAST(NULLIF(TRIM(Years_as_Customer), '') AS REAL) AS years_as_customer,
    CAST(NULLIF(TRIM(Num_of_Purchases), '') AS REAL) AS num_of_purchases,
    CAST(NULLIF(TRIM(Average_Transaction_Amount), '') AS REAL) AS average_transaction_amount,
    CAST(NULLIF(TRIM(Num_of_Returns), '') AS REAL) AS num_of_returns,
    CAST(NULLIF(TRIM(Num_of_Support_Contacts), '') AS REAL) AS num_of_support_contacts,
    CAST(NULLIF(TRIM(Satisfaction_Score), '') AS REAL) AS satisfaction_score,
    CAST(NULLIF(T

## Data-quality gate

In [9]:
build_database()           # raises DataQualityError if any check fails
import sqlite3
with sqlite3.connect(configs.DB_PATH) as con:
    checks = run_quality_checks(con)
checks

DatabaseError: Execution failed on sql 'SELECT 'null_engineered_features' AS check_name,
       COUNT(*)                   AS value,
       COUNT(*)                   AS violations
FROM customer_features
WHERE spend_per_year IS NULL
   OR purchases_per_year IS NULL
   OR return_rate IS NULL
   OR support_per_purchase IS NULL
   OR purchase_overdue_ratio IS NULL
   OR spend_to_income IS NULL': no such column: spend_to_income

Suggested retail checks for `sql/02_data_quality_checks.sql`: unique `customer_id`; no NULLs in model
features; `satisfaction_score` within its valid range; `target_churn` in (0, 1); no negative
spend/purchases/returns; `years_as_customer`, `num_of_purchases` and `annual_income` > 0 (or handled).

In [ ]:
run_named_query("kpi_overview")  # a first look through the BI layer

## Outliers

In [ ]:
feats = pd.read_csv(configs.FEATURES_CSV)
rows = []
for col in configs.NUMERIC_FEATURES:
    q1, q3 = feats[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n = ((feats[col] < q1 - 1.5 * iqr) | (feats[col] > q3 + 1.5 * iqr)).sum()
    rows.append({"column": col, "min": feats[col].min(), "max": feats[col].max(),
                 "skew": feats[col].skew().round(2), "iqr_outliers": n})
pd.DataFrame(rows)

In [ ]:
# engineered features must be finite
bad = np.isinf(feats[configs.NUMERIC_FEATURES]).sum() + feats[configs.NUMERIC_FEATURES].isna().sum()
bad[bad > 0]

*Interpretation (fill in after running):* money columns (`total_spend`, `annual_income`,
`spend_per_year`) are usually right-skewed with real high-value customers - keep them, don't drop
them as outliers. Tree models handle skew; a standardised logistic regression may want a log transform.

## Class balance

In [ ]:
feats[configs.TARGET].value_counts(normalize=True).rename("share")

*Interpretation (fill in after running):* state the churn rate and how imbalanced it is. If it is far from
50/50, accuracy is misleading, which is why this project judges models on ROC-AUC, PR-AUC and money.